# Using Jupyter Notebooks
:label:`sec_jupyter`


This section describes how to edit and run the code
in each section of this book
using the Jupyter Notebook. Make sure you have
installed Jupyter and downloaded the
code as described in
:ref:`chap_installation`.
If you want to know more about Jupyter see the excellent tutorial in
their [documentation](https://jupyter.readthedocs.io/en/latest/).


## Editing and Running the Code Locally

Suppose that the local path of the book's code is `xx/yy/d2l-en/`. Use the shell to change the directory to this path (`cd xx/yy/d2l-en`) and run the command `jupyter notebook`. If your browser does not do this automatically, open http://localhost:8888 and you will see the interface of Jupyter and all the folders containing the code of the book, as shown in :numref:`fig_jupyter00`.

![The folders containing the code of this book.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter00.png?raw=1)
:width:`600px`
:label:`fig_jupyter00`


You can access the notebook files by clicking on the folder displayed on the webpage.
They usually have the suffix ".ipynb".
For the sake of brevity, we create a temporary "test.ipynb" file.
The content displayed after you click it is
shown in :numref:`fig_jupyter01`.
This notebook includes a markdown cell and a code cell. The content in the markdown cell includes "This Is a Title" and "This is text.".
The code cell contains two lines of Python code.

![Markdown and code cells in the "text.ipynb" file.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter01.png?raw=1)
:width:`600px`
:label:`fig_jupyter01`


Double click on the markdown cell to enter edit mode.
Add a new text string "Hello world." at the end of the cell, as shown in :numref:`fig_jupyter02`.

![Edit the markdown cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter02.png?raw=1)
:width:`600px`
:label:`fig_jupyter02`


As demonstrated in :numref:`fig_jupyter03`,
click "Cell" $\rightarrow$ "Run Cells" in the menu bar to run the edited cell.

![Run the cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter03.png?raw=1)
:width:`600px`
:label:`fig_jupyter03`

After running, the markdown cell is shown in :numref:`fig_jupyter04`.

![The markdown cell after running.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter04.png?raw=1)
:width:`600px`
:label:`fig_jupyter04`


Next, click on the code cell. Multiply the elements by 2 after the last line of code, as shown in :numref:`fig_jupyter05`.

![Edit the code cell.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter05.png?raw=1)
:width:`600px`
:label:`fig_jupyter05`


You can also run the cell with a shortcut ("Ctrl + Enter" by default) and obtain the output result from :numref:`fig_jupyter06`.

![Run the code cell to obtain the output.](https://github.com/d2l-ai/d2l-en-colab/blob/master/img/jupyter06.png?raw=1)
:width:`600px`
:label:`fig_jupyter06`


When a notebook contains more cells, we can click "Kernel" $\rightarrow$ "Restart & Run All" in the menu bar to run all the cells in the entire notebook. By clicking "Help" $\rightarrow$ "Edit Keyboard Shortcuts" in the menu bar, you can edit the shortcuts according to your preferences.

## Advanced Options

Beyond local editing two things are quite important: editing the notebooks in the markdown format and running Jupyter remotely.
The latter matters when we want to run the code on a faster server.
The former matters since Jupyter's native ipynb format stores a lot of auxiliary data that is
irrelevant to the content,
mostly related to how and where the code is run.
This is confusing for Git, making
reviewing contributions very difficult.
Fortunately there is an alternative---native editing in the markdown format.

### Markdown Files in Jupyter

If you wish to contribute to the content of this book, you need to modify the
source file (md file, not ipynb file) on GitHub.
Using the notedown plugin we
can modify notebooks in the md format directly in Jupyter.


First, install the notedown plugin, run the Jupyter Notebook, and load the plugin:

```
pip install d2l-notedown  # You may need to uninstall the original notedown.
jupyter notebook --NotebookApp.contents_manager_class='notedown.NotedownContentsManager'
```

You may also turn on the notedown plugin by default whenever you run the Jupyter Notebook.
First, generate a Jupyter Notebook configuration file (if it has already been generated, you can skip this step).

```
jupyter notebook --generate-config
```

Then, add the following line to the end of the Jupyter Notebook configuration file (for Linux or macOS, usually in the path `~/.jupyter/jupyter_notebook_config.py`):

```
c.NotebookApp.contents_manager_class = 'notedown.NotedownContentsManager'
```

After that, you only need to run the `jupyter notebook` command to turn on the notedown plugin by default.

### Running Jupyter Notebooks on a Remote Server

Sometimes, you may want to run Jupyter notebooks on a remote server and access it through a browser on your local computer. If Linux or macOS is installed on your local machine (Windows can also support this function through third-party software such as PuTTY), you can use port forwarding:

```
ssh myserver -L 8888:localhost:8888
```

The above string `myserver` is the address of the remote server.
Then we can use http://localhost:8888 to access the remote server `myserver` that runs Jupyter notebooks. We will detail on how to run Jupyter notebooks on AWS instances
later in this appendix.

### Timing

We can use the `ExecuteTime` plugin to time the execution of each code cell in Jupyter notebooks.
Use the following commands to install the plugin:

```
pip install jupyter_contrib_nbextensions
jupyter contrib nbextension install --user
jupyter nbextension enable execute_time/ExecuteTime
```

## Summary

* Using the Jupyter Notebook tool, we can edit, run, and contribute to each section of the book.
* We can run Jupyter notebooks on remote servers using port forwarding.


## Exercises

1. Edit and run the code in this book with the Jupyter Notebook on your local machine.
1. Edit and run the code in this book with the Jupyter Notebook *remotely* via port forwarding.
1. Compare the running time of the operations $\mathbf{A}^\top \mathbf{B}$ and $\mathbf{A} \mathbf{B}$ for two square matrices in $\mathbb{R}^{1024 \times 1024}$. Which one is faster?


[Discussions](https://discuss.d2l.ai/t/421)


In [4]:
import numpy as np
from copy import deepcopy
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import differential_evolution, minimize


# ============================================================
# Utility functions
# ============================================================

def sqdist(X1, X2):
    """
    Pairwise squared Euclidean distance.
    MATLAB version expects X1, X2 as d x n matrices.
    """
    X1 = np.atleast_2d(np.asarray(X1, dtype=float))
    X2 = np.atleast_2d(np.asarray(X2, dtype=float))

    D = (
        np.sum(X2 * X2, axis=0)[None, :]
        + np.sum(X1 * X1, axis=0)[:, None]
        - 2.0 * (X1.T @ X2)
    )
    D[D < 0.0] = 0.0
    return D


def prior_gauss(mu, s2, x):
    lp = -((x - mu) ** 2) / (2.0 * s2) - 0.5 * np.log(2.0 * np.pi * s2)
    dlp = -(x - mu) / s2
    return lp, dlp


def stable_cholesky(K, max_tries=8):
    """
    Cholesky factorization with increasing jitter for numerical stability.
    Returns (cho_factor_output, jitter_used)
    """
    K = np.asarray(K, dtype=float)
    n = K.shape[0]
    jitter = 0.0

    for _ in range(max_tries):
        try:
            cf = cho_factor(K + jitter * np.eye(n), lower=True, check_finite=False)
            return cf, jitter
        except np.linalg.LinAlgError:
            jitter = 1e-10 if jitter == 0.0 else jitter * 10.0

    raise np.linalg.LinAlgError("Cholesky decomposition failed even after jittering.")


def inv_chol(A):
    cf, _ = stable_cholesky(A)
    return cho_solve(cf, np.eye(A.shape[0]))


def dominates(a, b):
    """
    Minimization dominance:
    a dominates b if a <= b in all objectives and < in at least one.
    """
    return np.all(a <= b) and np.any(a < b)


def pareto_front_mask(F):
    """
    Boolean mask of non-dominated points for minimization.
    O(N^2), which is fine for moderate population sizes.
    """
    F = np.asarray(F, dtype=float)
    n = F.shape[0]
    mask = np.ones(n, dtype=bool)

    for i in range(n):
        if not mask[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            if dominates(F[j], F[i]):
                mask[i] = False
                break
    return mask


# ============================================================
# Option handling
# ============================================================

def set_option_structure(opt, X, Y):
    """
    Python equivalent of MATLAB set_option_structure(Opt, X, Y)
    using nested dictionaries.
    """
    opt = deepcopy(opt) if opt is not None else {}

    opt.setdefault("pop", 100)
    opt.setdefault("Generation", 100)
    opt.setdefault("NoOfBachSequential", 1)
    opt.setdefault("maxeval", 1)

    opt.setdefault("Gen", {})
    opt["Gen"]["NoOfGPs"] = Y.shape[1]
    opt["Gen"]["NoOfInputDim"] = X.shape[1]

    gp_list = opt.get("GP", [])
    while len(gp_list) < opt["Gen"]["NoOfGPs"]:
        gp_list.append({})

    for i in range(opt["Gen"]["NoOfGPs"]):
        gp = gp_list[i]
        gp.setdefault("matern", 5)            # 1 / 3 / 5 / np.inf
        gp["cov"] = gp.get("matern", 5)
        gp.setdefault("nSpectralpoints", 500)
        gp.setdefault("fun_eval", 50)
        gp.setdefault("noiselimit", 0.0)
        gp.setdefault("var", 10.0)

        gp["h1"] = opt["Gen"]["NoOfInputDim"] + 1   # D lengthscales + 1 magnitude
        gp["h2"] = 1                                # noise

        gp.setdefault("priorlik", (-6.0, gp["var"]))
        gp.setdefault("priorcov", (0.0, gp["var"]))

        if "hyp" not in gp:
            gp["hyp"] = {}
        gp["hyp"].setdefault("cov", np.zeros(gp["h1"], dtype=float))
        gp["hyp"].setdefault("lik", float(np.log(1e-2)))

    opt["GP"] = gp_list
    return opt


# ============================================================
# Scaling
# ============================================================

def scale_variables(X, Y, lb, ub, opt):
    """
    Scale inputs to [0,1] and outputs to zero mean/unit variance.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    Xnew = (X - lb) / (ub - lb)

    ddof = 1 if Y.shape[0] > 1 else 0
    y_mean = np.mean(Y, axis=0)
    y_std = np.std(Y, axis=0, ddof=ddof)
    y_std = np.where(y_std < 1e-12, 1.0, y_std)

    Ynew = (Y - y_mean) / y_std
    return Xnew, Ynew


# ============================================================
# GP kernel / likelihood
# ============================================================

def cov_matern_anisotropic(matern, hyp_cov, sqrtK, expnK, dsq_M, X, i=None):
    """
    Python version of covMaternanisotropic.

    Parameters
    ----------
    matern : 1, 3, 5, or np.inf
    hyp_cov : array-like, length D+1
        [log(ell_1), ..., log(ell_D), log(sf)]
    sqrtK, expnK : intermediate matrices from likelihood computation
    dsq_M : block matrix of scaled squared distances, shape (n, n*D)
    X : input matrix, shape (n, D)
    i : derivative index (0-based); if None, return covariance matrix
    """
    n, D = X.shape
    sf2 = np.exp(2.0 * hyp_cov[D])

    if i is None:
        if matern == 3:
            t = sqrtK
            m = (1.0 + t) * expnK
        elif matern == 1:
            m = expnK
        elif matern == 5:
            t = sqrtK
            m = (1.0 + t * (1.0 + t / 3.0)) * expnK
        elif np.isinf(matern):
            m = expnK
        else:
            raise ValueError(f"Unsupported Matern parameter: {matern}")
        return sf2 * m

    # derivative branch
    if i < D:
        Ki = dsq_M[:, i * n:(i + 1) * n]

        if matern == 3:
            dm = expnK
        elif matern == 1:
            t = sqrtK
            dm = np.divide(expnK, t, out=np.zeros_like(expnK), where=t > 1e-15)
        elif matern == 5:
            t = sqrtK
            dm = ((1.0 + t) / 3.0) * expnK
        elif np.isinf(matern):
            dm = -0.5 * expnK
        else:
            raise ValueError(f"Unsupported Matern parameter: {matern}")

        K = sf2 * dm * Ki
        K[Ki < 1e-12] = 0.0
        return K

    elif i == D:
        if matern == 3:
            t = sqrtK
            m = (1.0 + t) * expnK
        elif matern == 1:
            m = expnK
        elif matern == 5:
            t = sqrtK
            m = (1.0 + t * (1.0 + t / 3.0)) * expnK
        elif np.isinf(matern):
            m = expnK
        else:
            raise ValueError(f"Unsupported Matern parameter: {matern}")

        return 2.0 * sf2 * m

    else:
        raise IndexError("Invalid derivative index in cov_matern_anisotropic.")


def n_likelihood(hyp_var, Xnew, Ynew, K_M, gp):
    """
    Negative log-likelihood and gradient.
    Equivalent to MATLAB NLikelihood(...).
    """
    Xnew = np.asarray(Xnew, dtype=float)
    Ynew = np.asarray(Ynew, dtype=float).reshape(-1)

    n, D = Xnew.shape
    matern = gp["cov"]
    dist_factor = 1.0 if np.isinf(matern) else float(matern)

    h1 = gp["h1"]
    h2 = gp["h2"]

    hyp_cov = np.asarray(hyp_var[:h1], dtype=float)
    hyp_lik = np.asarray(hyp_var[h1:h1 + h2], dtype=float)

    ell = np.exp(hyp_cov[:D])
    sf2 = np.exp(2.0 * hyp_cov[D])

    # Build scaled distance matrix
    K = np.zeros((n, n), dtype=float)
    for d in range(D):
        K += K_M[:, d * n:(d + 1) * n] * (dist_factor / (ell[d] ** 2))

    if np.isinf(matern):
        expnK = np.exp(-0.5 * K)
        sqrtK = None
    else:
        sqrtK = np.sqrt(K)
        expnK = np.exp(-sqrtK)

    if matern == 3:
        m = (1.0 + sqrtK) * expnK
    elif matern == 1:
        m = expnK
    elif matern == 5:
        m = (1.0 + sqrtK * (1.0 + sqrtK / 3.0)) * expnK
    elif np.isinf(matern):
        m = expnK
    else:
        raise ValueError(f"Unsupported Matern parameter: {matern}")

    sn2 = np.exp(2.0 * hyp_lik[0])
    K = sf2 * m + np.eye(n) * sn2
    K = 0.5 * (K + K.T)

    # Inverse and logdet
    cf, _ = stable_cholesky(K)
    invK = cho_solve(cf, np.eye(n))
    L = cf[0]
    logDetK = 2.0 * np.sum(np.log(np.abs(np.diag(L))))

    # Priors
    logprior = 0.0
    dlogpriorcov = np.zeros(h1, dtype=float)
    for i in range(h1):
        lp, dlp = prior_gauss(gp["priorcov"][0], gp["priorcov"][1], hyp_cov[i])
        logprior += lp
        dlogpriorcov[i] = dlp

    dlogpriorlik = np.zeros(h2, dtype=float)
    for i in range(h2):
        lp, dlp = prior_gauss(gp["priorlik"][0], gp["priorlik"][1], hyp_lik[i])
        logprior += lp
        dlogpriorlik[i] = dlp

    # NLL
    NLL = (
        0.5 * n * np.log(2.0 * np.pi)
        + 0.5 * logDetK
        + 0.5 * (Ynew.T @ invK @ Ynew)
        - logprior
    )

    # Gradient
    dsq_M = np.zeros((n, n * D), dtype=float)
    for d in range(D):
        dsq_M[:, d * n:(d + 1) * n] = K_M[:, d * n:(d + 1) * n] * (dist_factor / (ell[d] ** 2))

    c = invK @ Ynew

    dNLL_cov = np.zeros(h1, dtype=float)
    for i in range(h1):
        dK = cov_matern_anisotropic(matern, hyp_cov, sqrtK, expnK, dsq_M, Xnew, i=i)
        b = invK @ dK
        dNLL_cov[i] = 0.5 * np.trace(b) - 0.5 * (Ynew.T @ b @ c)

    dNLL_lik = np.zeros(h2, dtype=float)
    for i in range(h2):
        dK = 2.0 * np.exp(2.0 * hyp_lik[i]) * np.eye(n)
        b = invK @ dK
        dNLL_lik[i] = 0.5 * np.trace(b) - 0.5 * (Ynew.T @ b @ c)

    dNLL = np.concatenate([dNLL_cov, dNLL_lik]) - np.concatenate([dlogpriorcov, dlogpriorlik])

    return float(NLL), dNLL


def training_of_gp(Xnew, Ynew, gp):
    """
    Python replacement for:
        Direct(...) + fmincon(...)
    using:
        differential_evolution(...) + minimize(..., method='L-BFGS-B')
    """
    Xnew = np.asarray(Xnew, dtype=float)
    Ynew = np.asarray(Ynew, dtype=float).reshape(-1)

    n, D = Xnew.shape
    h1 = gp["h1"]
    h2 = gp["h2"]

    # Build block squared-distance matrix
    K_M = np.zeros((n, n * D), dtype=float)
    a = Xnew.T
    for i in range(D):
        K_M[:, i * n:(i + 1) * n] = sqdist(a[i:i+1, :], a[i:i+1, :])

    lower = np.full(h1 + h2, np.log(np.sqrt(1e-3)), dtype=float)
    upper = np.full(h1 + h2, np.log(np.sqrt(1e3)), dtype=float)
    lower[-1] = -6.0
    upper[-1] = gp["noiselimit"]

    bounds = list(zip(lower, upper))

    def fun(x):
        return n_likelihood(x, Xnew, Ynew, K_M, gp)[0]

    def jac(x):
        return n_likelihood(x, Xnew, Ynew, K_M, gp)[1]

    # Global search (replacement for DIRECT)
    de_res = differential_evolution(
        fun,
        bounds=bounds,
        maxiter=max(15, gp["fun_eval"]),
        popsize=10,
        polish=False,
        updating="deferred",
        workers=1,
        seed=None,
    )

    # Local refinement (replacement for fmincon)
    ls_res = minimize(
        fun=fun,
        x0=de_res.x,
        jac=jac,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": 2000, "ftol": 1e-12, "gtol": 1e-8},
    )

    xopt = ls_res.x if ls_res.success else de_res.x

    gp_hyp = {
        "cov": np.asarray(xopt[:h1], dtype=float),
        "lik": float(xopt[h1]),
    }
    return gp_hyp


# ============================================================
# Posterior / mean samples via random Fourier features
# ============================================================

def sample_multivariate_t_identity(df, n_samples, dim):
    """
    Sample from multivariate Student-t with identity scale matrix.
    """
    z = np.random.randn(n_samples, dim)
    g = np.random.chisquare(df, size=(n_samples, 1)) / df
    return z / np.sqrt(g)


def posterior_sample(Xnew, Ynew, gp):
    """
    Python translation of posterior_sample(...)
    """
    Xnew = np.asarray(Xnew, dtype=float)
    Ynew = np.asarray(Ynew, dtype=float).reshape(-1)

    nSpectralpoints = gp["nSpectralpoints"]
    n, D = Xnew.shape

    ell = np.exp(gp["hyp"]["cov"][:D])
    sf2 = np.exp(2.0 * gp["hyp"]["cov"][D])
    sn2 = np.exp(2.0 * gp["hyp"]["lik"])

    # W and b sampling
    if not np.isinf(gp["cov"]):
        W = sample_multivariate_t_identity(gp["cov"], nSpectralpoints, D) * (1.0 / ell)[None, :]
    else:
        W = np.random.randn(nSpectralpoints, D) * (1.0 / ell)[None, :]

    b = 2.0 * np.pi * np.random.rand(nSpectralpoints, 1)

    # phi
    phi = np.sqrt(2.0 * sf2 / nSpectralpoints) * np.cos(W @ Xnew.T + b)

    # theta posterior
    A = phi @ phi.T + sn2 * np.eye(nSpectralpoints)
    invA = inv_chol(A)

    mu_theta = invA @ phi @ Ynew
    cov_theta = sn2 * invA
    cov_theta = 0.5 * (cov_theta + cov_theta.T) + 1e-10 * np.eye(nSpectralpoints)

    theta = np.random.multivariate_normal(mean=mu_theta, cov=cov_theta)

    def f(x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        phi_x = np.sqrt(2.0 * sf2 / nSpectralpoints) * np.cos(W @ x.T + b)
        return (theta @ phi_x).reshape(-1)

    return f


def mean_sample(Xnew, Ynew, gp):
    """
    Python translation of mean_sample(...)
    """
    Xnew = np.asarray(Xnew, dtype=float)
    Ynew = np.asarray(Ynew, dtype=float).reshape(-1)

    nSpectralpoints = gp["nSpectralpoints"]
    n, D = Xnew.shape

    ell = np.exp(gp["hyp"]["cov"][:D])
    sf2 = np.exp(2.0 * gp["hyp"]["cov"][D])
    sn2 = np.exp(2.0 * gp["hyp"]["lik"])

    if not np.isinf(gp["cov"]):
        W = sample_multivariate_t_identity(gp["cov"], nSpectralpoints, D) * (1.0 / ell)[None, :]
    else:
        W = np.random.randn(nSpectralpoints, D) * (1.0 / ell)[None, :]

    b = 2.0 * np.pi * np.random.rand(nSpectralpoints, 1)

    phi = np.sqrt(2.0 * sf2 / nSpectralpoints) * np.cos(W @ Xnew.T + b)

    A = phi @ phi.T + sn2 * np.eye(nSpectralpoints)
    invA = inv_chol(A)
    mu_theta = invA @ phi @ Ynew

    def f(x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        phi_x = np.sqrt(2.0 * sf2 / nSpectralpoints) * np.cos(W @ x.T + b)
        return (mu_theta @ phi_x).reshape(-1)

    return f


# ============================================================
# Lightweight NSGA-II-style evolutionary search
# (used instead of external MATLAB nsga2)
# ============================================================

def non_dominated_sort(F):
    F = np.asarray(F, dtype=float)
    n = F.shape[0]

    S = [[] for _ in range(n)]
    dom_count = np.zeros(n, dtype=int)
    fronts = [[]]

    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            if dominates(F[p], F[q]):
                S[p].append(q)
            elif dominates(F[q], F[p]):
                dom_count[p] += 1

        if dom_count[p] == 0:
            fronts[0].append(p)

    i = 0
    while fronts[i]:
        next_front = []
        for p in fronts[i]:
            for q in S[p]:
                dom_count[q] -= 1
                if dom_count[q] == 0:
                    next_front.append(q)
        i += 1
        fronts.append(next_front)

    return [np.array(fr, dtype=int) for fr in fronts[:-1]]


def crowding_distance(F, front):
    front = np.asarray(front, dtype=int)
    n = len(front)
    d = np.zeros(n, dtype=float)

    if n == 0:
        return d
    if n <= 2:
        d[:] = np.inf
        return d

    vals = F[front]
    m = vals.shape[1]

    for j in range(m):
        order = np.argsort(vals[:, j])
        d[order[0]] = np.inf
        d[order[-1]] = np.inf

        fmin = vals[order[0], j]
        fmax = vals[order[-1], j]
        if abs(fmax - fmin) < 1e-15:
            continue

        for k in range(1, n - 1):
            if np.isinf(d[order[k]]):
                continue
            d[order[k]] += (vals[order[k + 1], j] - vals[order[k - 1], j]) / (fmax - fmin)

    return d


def better_index(i, j, rank, crowd):
    if rank[i] < rank[j]:
        return i
    if rank[j] < rank[i]:
        return j
    if crowd[i] > crowd[j]:
        return i
    if crowd[j] > crowd[i]:
        return j
    return i if np.random.rand() < 0.5 else j


def blend_crossover(p1, p2, lb, ub, prob=0.9, alpha=0.5):
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    if np.random.rand() > prob:
        return p1.copy(), p2.copy()

    gamma = (1.0 + 2.0 * alpha) * np.random.rand(p1.size) - alpha
    c1 = (1.0 - gamma) * p1 + gamma * p2
    c2 = gamma * p1 + (1.0 - gamma) * p2

    c1 = np.clip(c1, lb, ub)
    c2 = np.clip(c2, lb, ub)
    return c1, c2


def mutate(x, lb, ub, prob=None, sigma=0.1):
    x = np.asarray(x, dtype=float).copy()
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    D = x.size
    if prob is None:
        prob = 1.0 / D

    mask = np.random.rand(D) < prob
    if np.any(mask):
        x[mask] += sigma * (ub[mask] - lb[mask]) * np.random.randn(np.sum(mask))
    x = np.clip(x, lb, ub)
    return x


def moea_minimize(objfun, D, n_obj, popsize, generations, lb, ub):
    """
    Lightweight multi-objective evolutionary minimizer.
    Designed to replace MATLAB nsga2(...) in a self-contained way.
    """
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    X = lb + (ub - lb) * np.random.rand(popsize, D)
    F = objfun(X)

    for _ in range(generations):
        fronts = non_dominated_sort(F)
        rank = np.empty(popsize, dtype=int)
        crowd = np.zeros(popsize, dtype=float)

        for r, fr in enumerate(fronts):
            rank[fr] = r
            crowd[fr] = crowding_distance(F, fr)

        # Tournament selection
        mating = []
        for _ in range(popsize):
            i, j = np.random.randint(0, popsize, size=2)
            mating.append(better_index(i, j, rank, crowd))
        mating = np.asarray(mating, dtype=int)

        # Variation
        offspring = []
        for i in range(0, popsize, 2):
            p1 = X[mating[i]]
            p2 = X[mating[(i + 1) % popsize]]
            c1, c2 = blend_crossover(p1, p2, lb, ub, prob=0.9, alpha=0.5)
            c1 = mutate(c1, lb, ub, sigma=0.1)
            c2 = mutate(c2, lb, ub, sigma=0.1)
            offspring.append(c1)
            offspring.append(c2)

        Xo = np.asarray(offspring[:popsize], dtype=float)
        Fo = objfun(Xo)

        # Environmental selection
        Xc = np.vstack([X, Xo])
        Fc = np.vstack([F, Fo])

        fronts = non_dominated_sort(Fc)

        new_X = []
        new_F = []

        for fr in fronts:
            if len(new_X) + len(fr) <= popsize:
                for idx in fr:
                    new_X.append(Xc[idx])
                    new_F.append(Fc[idx])
            else:
                cd = crowding_distance(Fc, fr)
                order = np.argsort(-cd)   # descending crowding
                need = popsize - len(new_X)
                chosen = fr[order[:need]]
                for idx in chosen:
                    new_X.append(Xc[idx])
                    new_F.append(Fc[idx])
                break

        X = np.asarray(new_X, dtype=float)
        F = np.asarray(new_F, dtype=float)

    return X, F


# ============================================================
# Pareto optimization of GP samples / means
# ============================================================

def pareto_objective(X, sample_functions):
    X = np.atleast_2d(np.asarray(X, dtype=float))
    F = np.column_stack([f(X) for f in sample_functions])
    return F


def find_sample_pareto(opt):
    D = opt["Gen"]["NoOfInputDim"]
    n_obj = opt["Gen"]["NoOfGPs"]

    objfun = lambda x: pareto_objective(x, opt["Sample"])
    Xpop, Fpop = moea_minimize(
        objfun=objfun,
        D=D,
        n_obj=n_obj,
        popsize=opt["pop"],
        generations=opt["Generation"],
        lb=np.zeros(D),
        ub=np.ones(D),
    )

    sample_nadir = np.max(Fpop, axis=0)
    return Fpop, Xpop, sample_nadir


def find_mean_pareto(opt):
    D = opt["Gen"]["NoOfInputDim"]
    n_obj = opt["Gen"]["NoOfGPs"]

    objfun = lambda x: pareto_objective(x, opt["Mean"])
    Xpop, Fpop = moea_minimize(
        objfun=objfun,
        D=D,
        n_obj=n_obj,
        popsize=opt["pop"],
        generations=opt["Generation"],
        lb=np.zeros(D),
        ub=np.ones(D),
    )

    return Fpop, Xpop


# ============================================================
# Hypervolume
# ============================================================

def remove_points_above_reference(Afront, r):
    Afront = np.asarray(Afront, dtype=float)
    r = np.asarray(r, dtype=float)
    if Afront.size == 0:
        return Afront.reshape(0, r.size)
    mask = np.all(Afront <= r[None, :], axis=1)
    return Afront[mask]


def hypervolume_2d(Yfront, r):
    """
    Exact hypervolume for 2 objectives, minimization setting.
    """
    A = remove_points_above_reference(Yfront, r)
    if A.size == 0:
        return 0.0

    A = A[pareto_front_mask(A)]
    A = A[np.argsort(A[:, 0])]

    hv = 0.0
    prev_y = r[1]

    for x, y in A:
        if y < prev_y:
            hv += max(0.0, r[0] - x) * max(0.0, prev_y - y)
            prev_y = y

    return float(hv)


def hypervolume_mc(Yfront, r, n_samples=10000):
    """
    Monte Carlo hypervolume for 3+ objectives (or fallback).
    """
    A = remove_points_above_reference(Yfront, r)
    if A.size == 0:
        return 0.0

    A = A[pareto_front_mask(A)]
    lower = np.min(A, axis=0)

    if np.any(r <= lower):
        return 0.0

    samples = lower + np.random.rand(n_samples, A.shape[1]) * (r - lower)

    # A point in samples is dominated if any point in A is <= it in all coordinates
    dominated = np.any(np.all(A[:, None, :] <= samples[None, :, :], axis=2), axis=0)
    hv = np.prod(r - lower) * np.mean(dominated)
    return float(hv)


def hypervolume(Yfront, r):
    m = Yfront.shape[1]
    if m == 2:
        return hypervolume_2d(Yfront, r)
    else:
        return hypervolume_mc(Yfront, r, n_samples=10000)


def hypervolume_improvement_index(Ynew, sample_nadir, sample_pareto, opt):
    """
    Python translation of hypervolume_improvement_index(...)
    """
    Ynew = np.asarray(Ynew, dtype=float)
    sample_pareto = np.asarray(sample_pareto, dtype=float)

    r = sample_nadir + 0.01 * (np.max(sample_pareto, axis=0) - np.min(sample_pareto, axis=0))
    index = []

    Ywork = Ynew.copy()
    hvY0 = None
    hv_improvement_last = None
    hvY = None

    for i in range(opt["NoOfBachSequential"]):
        Yfront = Ywork[pareto_front_mask(Ywork)]
        hvY = hypervolume(Yfront, r)

        if i == 0:
            hvY0 = hvY

        hv_improvement = np.zeros(sample_pareto.shape[0], dtype=float)

        for k in range(sample_pareto.shape[0]):
            A = np.vstack([Ywork, sample_pareto[k]])
            Afront = A[pareto_front_mask(A)]
            hv = hypervolume(Afront, r)
            hv_improvement[k] = hv - hvY

        current_index = int(np.argmax(hv_improvement))
        Ywork = np.vstack([Ywork, sample_pareto[current_index]])
        index.append(current_index)
        hv_improvement_last = hv_improvement

    hv_imp = hv_improvement_last[index[-1]] + hvY - hvY0
    return np.asarray(index, dtype=int), float(hv_imp)


# ============================================================
# Main TSEMO function
# ============================================================

def tsemo(X, Y, lb, ub, opt=None):
    """
    Python translation of the MATLAB TSEMO(...) function.

    Parameters
    ----------
    X : array, shape (n, D)
    Y : array, shape (n, O)
    lb : array, shape (D,)
    ub : array, shape (D,)
    opt : dict
        options dictionary

    Returns
    -------
    xnewtrue : array, shape (batch, D)
        proposed next evaluation point(s) in original variable scale
    info : dict
        additional details (hv improvement, trained hyperparameters, etc.)
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    if X.ndim != 2:
        raise ValueError("X must be 2D: shape (n, D)")
    if Y.ndim != 2:
        raise ValueError("Y must be 2D: shape (n, O)")
    if X.shape[0] != Y.shape[0]:
        raise ValueError("X and Y must have the same number of rows")
    if X.shape[1] != lb.size or X.shape[1] != ub.size:
        raise ValueError("lb and ub must match the number of input dimensions")

    # Initialize options
    opt = set_option_structure(opt, X, Y)

    # Scale variables
    Xnew, Ynew = scale_variables(X, Y, lb, ub, opt)

    # Train one GP per objective
    for j in range(opt["Gen"]["NoOfGPs"]):
        gp_hyp = training_of_gp(Xnew, Ynew[:, j], opt["GP"][j])
        opt["GP"][j]["hyp"] = gp_hyp

    # Draw posterior samples of GPs
    opt["Sample"] = []
    for j in range(opt["Gen"]["NoOfGPs"]):
        f = posterior_sample(Xnew, Ynew[:, j], opt["GP"][j])
        opt["Sample"].append(f)

    # (Optional) mean functions, if you want later
    opt["Mean"] = []
    for j in range(opt["Gen"]["NoOfGPs"]):
        f = mean_sample(Xnew, Ynew[:, j], opt["GP"][j])
        opt["Mean"].append(f)

    # Determine Pareto front of function samples
    sample_pareto, sample_xpareto, sample_nadir = find_sample_pareto(opt)

    # Select point(s) with maximum hypervolume improvement
    index, hv_imp = hypervolume_improvement_index(Ynew, sample_nadir, sample_pareto, opt)

    xNew_scaled = sample_xpareto[index, :]
    xnewtrue = xNew_scaled * (ub - lb) + lb

    info = {
        "hv_imp": hv_imp,
        "index": index,
        "sample_pareto": sample_pareto,
        "sample_xpareto": sample_xpareto,
        "sample_nadir": sample_nadir,
        "opt": opt,
    }

    return xnewtrue, info

In [5]:
# Example: 2 inputs, 2 objectives
np.random.seed(123)

X = np.array([
    [0.1, 0.2],
    [0.4, 0.6],
    [0.8, 0.3],
    [0.7, 0.9],
])

Y = np.array([
    [1.2, 3.4],
    [0.9, 2.8],
    [1.5, 2.1],
    [0.7, 3.0],
])

lb = np.array([0.0, 0.0])
ub = np.array([1.0, 1.0])

opt = {
    "pop": 80,
    "Generation": 50,
    "NoOfBachSequential": 1,
    "GP": [
        {"matern": 5, "nSpectralpoints": 300, "fun_eval": 30},
        {"matern": 5, "nSpectralpoints": 300, "fun_eval": 30},
    ],
}

x_next, info = tsemo(X, Y, lb, ub, opt)

print("Next suggested point(s):")
print(x_next)

print("\nPredicted hypervolume improvement:")
print(info["hv_imp"])

Next suggested point(s):
[[1.         0.70851979]]

Predicted hypervolume improvement:
2.2269201496957964


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
import importlib.util

# -------------------------------------------
# Helper: Load external algorithm file (.py)
# -------------------------------------------
def load_algorithm(path):
    spec = importlib.util.spec_from_file_location("algo_module", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# -------------------------------------------
# Helper: Pareto Front
# -------------------------------------------
def pareto_front(points):
    n = points.shape[0]
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if all(points[j] <= points[i]) and any(points[j] < points[i]):
                is_pareto[i] = False
                break
    idx = np.where(is_pareto)[0]
    return points[idx], idx


# -------------------------------------------
# Monte‑Carlo Hypervolume
# -------------------------------------------
def hypervolume_mc(F, A, U, N=100000):
    F = np.array(F)
    A = np.array(A)
    U = np.array(U)

    samples = A + (U - A) * np.random.rand(N, 2)

    dominated = np.zeros(N, dtype=bool)
    for p in F:
        dominated |= np.all(samples >= p, axis=1)     # minimization assumption

    return np.mean(dominated)


# -------------------------------------------
# Main Experiment Script
# -------------------------------------------

def run_lactose_experiment(selectedfile):

    # =====================================
    # Generate Initial Dataset (LHS)
    # =====================================
    initial_sample_size = 20
    inputdim = 2

    sampler = qmc.LatinHypercube(d=inputdim)
    initialX = sampler.random(n=initial_sample_size)

    lowerbounds = np.array([0.5, 25])
    upperbounds = np.array([10, 100])

    conditions = initialX * (upperbounds - lowerbounds) + lowerbounds

    tR = conditions[:, 0]
    T = conditions[:, 1]
    ConcA = 1

    # Allocate storage
    data = {}
    data["x"] = []
    data["y"] = []
    data["opt"] = []
    data["hv"] = []

    # =====================================
    # Run initial LHC experiments
    # =====================================
    hv_list = []

    for i in range(len(conditions)):
        # Simulate reaction
        Cfinal, Yield, TimeData, ConcData = Lactose_Constants(
            ConcA, np.ones(4) * T[i], tR[i]
        )

        # Extract yields
        SM = Yield[0] * 100
        product = Yield[1] * 100
        side1 = Yield[2] * 100

        # Add measurement noise
        def add_noise(v):
            v2 = v + ((np.random.rand() - 0.5) * 0.5) + (v * ((np.random.rand() - 0.5) / 100))
            return float(min(max(v2, 1e-6), 100))

        SM = add_noise(SM)
        product = add_noise(product)
        side1 = add_noise(side1)

        # Calculate STY
        STY_val = STY_calc(product / 100, tR[i], ConcData[0][0], 342.30)

        # Objective transformation
        f1 = -np.log(product)
        f2 = np.log(side1)
        f = [f1, f2]

        # Update dataset
        data["x"].append(conditions[i])
        data["y"].append(f)

        # Compute Pareto front
        z = np.column_stack((
            np.exp(-np.array([row[0] for row in data["y"]])),
            -np.exp(np.array([row[1] for row in data["y"]]))
        ))

        opt, idxs = pareto_front(z)
        data["opt"].append(len(opt))

        # Hypervolume
        AU = np.array([0.1350, -62.0549])
        U = np.array([26.7043, -1e-6])
        hv = hypervolume_mc(opt, AU, U, N=100000) * 100
        data["hv"].append(hv)

        # Plot
        plt.clf()
        plt.plot(range(1, len(data["hv"]) + 1), data["hv"], linewidth=2)
        plt.title("Hypervolume")
        plt.xlabel("Experiment Number")
        plt.ylabel("Hypervolume")
        plt.pause(0.1)

    data["x"] = np.array(data["x"])
    data["y"] = np.array(data["y"])

    # =====================================
    # Sequential Optimization Loop
    # =====================================
    total_expts = len(data["x"])

    algo = load_algorithm(selectedfile)

    while total_expts < 100:

        # Run user-selected algorithm file
        new_conditions = algo.next_point(data)

        # Append new point
        conditions = np.vstack([conditions, new_conditions])

        # Run new experiment
        i = len(conditions) - 1

        Cfinal, Yield, TimeData, ConcData = Lactose_Constants(
            ConcA, np.ones(4) * conditions[i][1], conditions[i][0]
        )

        SM = add_noise(Yield[0] * 100)
        product = add_noise(Yield[1] * 100)
        side1 = add_noise(Yield[2] * 100)

        STY_val = STY_calc(product / 100, conditions[i][0], ConcData[0][0], 342.30)

        # Objective transform
        f1 = -np.log(product)
        f2 = np.log(side1)

        data["x"] = np.vstack([data["x"], conditions[i]])
        data["y"] = np.vstack([data["y"], [f1, f2]])

        total_expts = len(data["x"])

        # Pareto front
        z = np.column_stack((
            np.exp(-data["y"][:, 0]),
            -np.exp(data["y"][:, 1])
        ))

        opt, idxs = pareto_front(z)
        data["opt"].append(len(opt))

        # Hypervolume
        hv = hypervolume_mc(opt, AU, U, N=100000) * 100
        data["hv"].append(hv)

        plt.clf()
        plt.plot(range(1, len(data["hv"]) + 1), data["hv"], linewidth=2)
        plt.title("Hypervolume")
        plt.xlabel("Experiment Number")
        plt.ylabel("Hypervolume")
        plt.pause(0.1)

    return data

In [7]:
import numpy as np

# Global variables (same names as MATLAB)
Ea = None
ko = None
discrete_v = None
n = None
flowrate = None


def Lactose_Equations(t, y):
    """
    Python equivalent of MATLAB Lactose_Equations.

    Inputs:
        t : time (ignored)
        y : state vector [concentrations (3*n), temperatures (n-1)]

    Output:
        dydt : time derivative of state vector
    """

    global Ea, ko, discrete_v, n, flowrate

    # ------------------------------------------
    # Reshape concentration block
    # ------------------------------------------
    # MATLAB: conc = reshape(y(1:n*3), n, [])
    conc = y[: n * 3].reshape(n, 3)

    # Temperature block:
    # MATLAB: T = y(n*3+1 : n*3+n-1)
    T = y[n * 3 : n * 3 + (n - 1)]

    # ------------------------------------------
    # Compute rate constants (vectorized)
    # ------------------------------------------
    # MATLAB:
    #   ksq = [ko;ko;ko;ko]
    #   Easq = [Ea;Ea;Ea;Ea]
    #   Tsq = [T T T]
    #
    # Equivalent in Python:
    ksq = np.tile(ko, (n - 1, 1))         # shape ((n-1) × 3)
    Easq = np.tile(Ea, (n - 1, 1))        # shape ((n-1) × 3)
    Tsq = np.tile(T.reshape(-1, 1), (1, 3))   # shape ((n-1) × 3)

    # Arrhenius equation:
    k = ksq * np.exp(-Easq / (8.314 * (Tsq + 273.15)))

    # ------------------------------------------
    # Species concentrations
    # ------------------------------------------
    A = conc[:, 0]
    B = conc[:, 1]
    C = conc[:, 2]

    # ------------------------------------------
    # Compute dA/dt, dB/dt, dC/dt
    # ------------------------------------------

    # Flow term using finite difference:
    # diff(A) / diff(discrete_v)
    dA_flow = flowrate * np.diff(A) / np.diff(discrete_v)
    dB_flow = flowrate * np.diff(B) / np.diff(discrete_v)
    dC_flow = flowrate * np.diff(C) / np.diff(discrete_v)

    # Reaction terms:
    # Use A(2:end) -> A[1:], B(2:end) -> B[1:], etc.
    rA = k[:, 0] * A[1:] + k[:, 1] * A[1:]
    rB = k[:, 3 - 3] * B[1:] - k[:, 0] * A[1:]      # careful: same indexing as MATLAB
    rC = k[:, 1] * A[1:] + k[:, 2] * B[1:]

    # MATLAB code:
    # dAdt = [0; -flow*diff(A)/diff(V) - (k1*A(2:end) + k2*A(2:end))]
    dAdt = np.zeros(n)
    dAdt[1:] = -dA_flow - rA

    # dBdt = [0; -flow*diff(B)/diff(V) - (k3*B(2:end) - k1*A(2:end))]
    dBdt = np.zeros(n)
    dBdt[1:] = -dB_flow - rB

    # dCdt = [0; -flow*diff(C)/diff(V) + (k2*A(2:end) + k3*B(2:end))]
    dCdt = np.zeros(n)
    dCdt[1:] = -dC_flow + rC

    # ------------------------------------------
    # Assemble dydt
    # MATLAB:
    # dydt = [reshape([dAdt dBdt dCdt],1,[])'; [0 0 0 0]'];
    # ------------------------------------------

    # Flatten dAdt, dBdt, dCdt in MATLAB column-major order
    conc_dot = np.column_stack([dAdt, dBdt, dCdt]).reshape(-1)

    # Temperature derivatives = zeros
    Tdot = np.zeros(n - 1)

    return np.concatenate([conc_dot, Tdot])